<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Weak_Force_Animation_Feynman_Diagrams.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Weak Force Animation Feynman Diagrams

This notebook generates a high-quality scientific visualization of **Weak Interaction** processes using Python's `matplotlib` and `ffmpeg`. Designed for educational storytelling and cinematic impact.

### 🔬 Scientific Scope
The animation covers key processes governed by the weak nuclear force:
1. **Muon Decay**: $\mu^- \to e^- + \bar{\nu}_e + \nu_\mu$ via $W^-$ exchange.
2. **Beta-Minus Decay**: Transformation of a neutron into a proton within a nucleus.
3. **LHC W Production**: Quark-antiquark annihilation ($u + \bar{d} \to W^+$).
4. **Z-Scattering**: Neutral current interaction showing no charge exchange.
5. **Pion Decay**: Most common decay mode of the charged pion ($\pi^-$).
6. **GIM Mechanism**: Visualizing how quantum interference suppresses flavor-changing neutral currents.

### 🎬 Visual & Technical Features
- **Motion Design**: Smooth ease-in/out trajectories and animated sinusoidal propagators for W/Z bosons.
- **Cinematic Effects**: Layered glow effects, vertex flashes, and dynamic particle labeling.
- **Physics Accuracy**: Correct antiquark notation ($\bar{q}$) and specific color-coding for antimatter.
- **Optimized Rendering**: Designed to run efficiently within Google Colab environments.

### 👤 Author Information
- **Creator**: Mugambi Ndwiga
- **Instagram**: [@craftsandengineering](https://www.instagram.com/craftsandengineering)

---
*Note: To run the animation, ensure FFmpeg is installed in the environment (default in Colab).*

---

In [12]:
"""
The Weak Force — Scientific Animation

Author:
    Mugambi Ndwiga

Instagram:
    @craftsandengineering

Description:
    Cinematic scientific visualization of weak interaction processes,
    including muon decay, beta decay, W boson production,
    neutrino scattering, pion decay, and the GIM mechanism.

Platform:
    Google Colab

Libraries:
    matplotlib, numpy
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import HTML
from base64 import b64encode
import google.colab.files

# --- Configuration ---
FPS = 30
SCENE_DUR = 7
NUM_SCENES = 8
TOTAL_FRAMES = FPS * SCENE_DUR * NUM_SCENES

COLORS = {
    'bg': '#050510', 'muon': '#4fc3f7', 'electron': '#81d4fa',
    'neutrino': '#b2ebf2', 'w_boson': '#ce93d8', 'z_boson': '#9575cd',
    'u_quark': '#ef9a9a', 'd_quark': '#a5d6a7', 'hadron': '#37474f',
    'hadron_p': '#546e7a', 'positron': '#ff8a80', 'white': '#ffffff',
    'antiquark': '#b3e5fc', 'glow': '#ce93d8'
}

# Setup Figure
plt.rcParams['text.usetex'] = False
fig, ax = plt.subplots(figsize=(12.8, 7.2), dpi=100)
fig.patch.set_facecolor(COLORS['bg'])
ax.set_facecolor(COLORS['bg'])
ax.set_xlim(-5, 5); ax.set_ylim(-3.5, 3.5); ax.axis('off')

# Permanent Watermark
watermark = ax.text(4.8, -3.3, "Mugambi Ndwiga | @craftsandengineering",
                    color='white', alpha=0.2, ha='right', fontsize=9)

# --- Helpers ---
def ease_io(t): return 0.5 - 0.5 * np.cos(np.pi * np.clip(t, 0, 1))

def get_wavy(p1, p2, frame, amp=0.12, freq=10):
    dist = np.hypot(p2[0]-p1[0], p2[1]-p1[1]) + 1e-9
    t_vals = np.linspace(0, 1, 60)
    x = np.linspace(p1[0], p2[0], 60)
    y = np.linspace(p1[1], p2[1], 60)
    norm = np.array([-(p2[1]-p1[1]), p2[0]-p1[0]]) / dist
    phase = frame * 0.4
    sine = amp * np.sin(2 * np.pi * freq * t_vals - phase)
    return x + sine * norm[0], y + sine * norm[1]

def draw_glow_line(x, y, color, alpha=1.0, lw=2):
    g1, = ax.plot(x, y, color=color, lw=lw*4, alpha=0.15*alpha)
    g2, = ax.plot(x, y, color=color, lw=lw, alpha=alpha)
    return [g1, g2]

artists = []

def clear_artists():
    global artists
    for a in artists:
        try: a.remove()
        except: pass
    artists = []

def update(frame):
    clear_artists()
    scene_idx = frame // (FPS * SCENE_DUR)
    rel_t = (frame % (FPS * SCENE_DUR)) / (FPS * SCENE_DUR)
    alpha = np.clip(rel_t*5, 0, 1) if rel_t < 0.2 else np.clip((1-rel_t)*5, 0, 1)

    if scene_idx == 0: # Title Card
        t1 = ax.text(0, 0.3, "The Weak Force", color='white', fontsize=35, ha='center', weight='bold', alpha=alpha)
        t2 = ax.text(0, -0.4, "Particle Transformations & Quantum Interactions", color='white', fontsize=16, ha='center', alpha=alpha*0.8)
        t3 = ax.text(0, -1.2, "By Mugambi Ndwiga | @craftsandengineering", color='white', fontsize=12, ha='center', alpha=alpha*0.6)
        artists.extend([t1, t2, t3])

    elif scene_idx == 1: # Muon Decay
        # Muon travel
        p = ease_io(rel_t * 2)
        mx, my = [-4, -1*p], [0, 0]
        artists.extend(draw_glow_line(mx, my, COLORS['muon'], alpha=alpha))
        artists.append(ax.text(-3.5, 0.3, r'$\mu^-$', color=COLORS['muon'], alpha=alpha, fontsize=15))

        if rel_t > 0.4:
            wx, wy = get_wavy([-1, 0], [0.5, -1.5], frame)
            artists.extend(draw_glow_line(wx, wy, COLORS['w_boson'], alpha=alpha))
            artists.append(ax.text(0.2, -0.6, r'$W^-$', color=COLORS['w_boson'], alpha=alpha))

            ep = ease_io((rel_t-0.5)*2)
            artists.extend(draw_glow_line([0.5, 0.5+ep*2], [-1.5, -1.5-ep*1], COLORS['electron'], alpha=alpha))
            artists.extend(draw_glow_line([0.5, 0.5+ep*2], [-1.5, -1.5+ep*1], COLORS['neutrino'], alpha=alpha*0.6))
            artists.append(ax.text(2.6, -2.8, r'$e^-$', color=COLORS['electron'], alpha=alpha))

        txt = ax.text(0, 2.5, "Weak force changes particle identity", color='white', ha='center', fontsize=16, alpha=alpha)
        artists.append(txt)

    elif scene_idx == 2: # Beta Decay
        blob = patches.Circle((0, 0), 1.8, color=COLORS['hadron'], alpha=0.3*alpha)
        ax.add_patch(blob); artists.append(blob)
        label = "Neutron (n)" if rel_t < 0.5 else "Proton (p+)"
        artists.append(ax.text(0, 2.2, label, color='white', ha='center', fontsize=18, alpha=alpha))

        # d -> u transition
        if rel_t > 0.4:
            wx, wy = get_wavy([0, 0], [2.5, -2], frame)
            artists.extend(draw_glow_line(wx, wy, COLORS['w_boson'], alpha=alpha))
            artists.append(ax.text(2.6, -2.2, r'$e^-$', color=COLORS['electron'], alpha=alpha))
            artists.append(ax.text(1.2, -0.8, r'$W^-$', color=COLORS['w_boson'], alpha=alpha))

    elif scene_idx == 3: # LHC Production
        p = ease_io(rel_t*2)
        # u + anti-d -> W+
        artists.extend(draw_glow_line([-4, -4+p*4], [0.5, 0.5], COLORS['u_quark'], alpha=alpha))
        artists.extend(draw_glow_line([4, 4-p*4], [0.5, 0.5], COLORS['antiquark'], alpha=alpha))
        artists.append(ax.text(-3.8, 0.8, r'$u$', color=COLORS['u_quark'], alpha=alpha))
        artists.append(ax.text(3.5, 0.8, r'$\bar{d}$', color=COLORS['antiquark'], alpha=alpha))

        if rel_t > 0.5:
            flash = patches.Circle((0, 0.5), 0.5, color='white', alpha=0.5*alpha)
            ax.add_patch(flash); artists.append(flash)
            wx, wy = get_wavy([0, 0.5], [0, 2.5], frame)
            artists.extend(draw_glow_line(wx, wy, COLORS['w_boson'], alpha=alpha))
            artists.append(ax.text(0.3, 1.5, r'$W^+$', color=COLORS['w_boson'], alpha=alpha))

    elif scene_idx == 4: # Z-Scattering
        artists.extend(draw_glow_line([-4, 4], [1.5, 1.5], COLORS['neutrino'], alpha=alpha*0.5))
        artists.extend(draw_glow_line([-4, 4], [-1.5, -1.5], COLORS['electron'], alpha=alpha))
        wx, wy = get_wavy([0, 1.5], [0, -1.5], frame)
        artists.extend(draw_glow_line(wx, wy, COLORS['z_boson'], alpha=alpha))
        artists.append(ax.text(0.3, 0, r'$Z^0$', color=COLORS['z_boson'], alpha=alpha))
        artists.append(ax.text(0, 2.5, "Neutral current: No charge exchange", color='white', ha='center', fontsize=16, alpha=alpha))

    elif scene_idx == 5: # Pion Decay
        pion = patches.Ellipse((0, 0), 1.5, 1, color=COLORS['hadron'], alpha=0.4*alpha)
        ax.add_patch(pion); artists.append(pion)
        artists.append(ax.text(0, 1.2, r'$\pi^-$ ($d\bar{u}$)', color='white', ha='center', alpha=alpha))
        if rel_t > 0.6:
            wx, wy = get_wavy([0, 0], [3, 0], frame)
            artists.extend(draw_glow_line(wx, wy, COLORS['w_boson'], alpha=alpha))
            artists.append(ax.text(3.2, 0.2, r'$\mu^-$', color=COLORS['muon'], alpha=alpha))

    elif scene_idx == 6: # GIM Cancellation
        artists.append(ax.text(0, 2.5, "GIM Mechanism: Quantum Interference", color='white', ha='center', fontsize=18, alpha=alpha))
        # Draw a box loop
        box = patches.Rectangle((-1.5, -1), 3, 2, fill=False, edgecolor='white', ls='--', alpha=0.3*alpha)
        ax.add_patch(box); artists.append(box)
        artists.append(ax.text(0, 0, r'$\sum amplitudes \approx 0$', color='white', ha='center', alpha=alpha))

    elif scene_idx == 7: # End Card
        e1 = ax.text(0, 0.5, "Created by Mugambi Ndwiga", color='white', fontsize=24, ha='center', alpha=alpha)
        e2 = ax.text(0, -0.2, "Instagram: @craftsandengineering", color='white', fontsize=16, ha='center', alpha=alpha)

    return artists

anim = FuncAnimation(fig, update, frames=TOTAL_FRAMES, blit=False)
writer = FFMpegWriter(fps=FPS, metadata=dict(artist='Mugambi Ndwiga'), bitrate=2500)
anim.save('weak_force.mp4', writer=writer)
plt.close()

with open('weak_force.mp4','rb') as f:
    data_url = "data:video/mp4;base64," + b64encode(f.read()).decode()
display(HTML(f'<video width=960 controls autoplay loop><source src="{data_url}" type="video/mp4"></video>'))
google.colab.files.download('weak_force.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>